In [5]:

# Introduzione ------------------------------------------------------------
library(tidyverse)
library(patchwork)
library(viridisLite)
library(viridis)

theme_set(theme_bw())

# Caricamento e pulizia iniziale
df <- read_csv("movie_statistic_dataset.csv", na = c("-",'NA','','\\N')) %>%
  mutate(
    movie_averageRating = as.numeric(movie_averageRating),
    `Production budget $` = as.numeric(`Production budget $`),
    `Worldwide gross $` = as.numeric(`Worldwide gross $`),
    runtime_minutes = as.numeric(runtime_minutes)
  )

# Valori nulli  -----------------------------------------------------------


# Conta i trattini per ogni colonna
colSums(is.na(df))
df_incompleti <- df[!complete.cases(df), ]
head(df_incompleti)
# Crea un sotto-dataset con solo i film che hanno dati mancanti
film_con_na <- df[!complete.cases(df), ]

# Visualizza i primi risultati
view(film_con_na)
glimpse(df)



# Analisi grafiche preliminari --------------------------------------------


ggplot(df, aes(x = movie_averageRating)) +
  geom_histogram(binwidth = 0.2, fill = "royalblue", color = "white") +
  geom_vline(aes(xintercept = mean(movie_averageRating)), color = "red", linetype = "dashed") +
  labs(title = "Distribuzione dei Rating dei Film", x = "Voto Medio", y = "Conteggio") +
  theme_bw()
df_genres <- df %>%
  separate_rows(genres, sep = ",") %>%
  filter(genres %in% unique(genres)) # Filtriamo i principali

ggplot(df_genres, aes(x = genres, y = movie_averageRating, fill = genres)) +
  geom_boxplot(outlier.alpha = 0.2) +
  coord_flip() + # Ruota il grafico per leggere meglio i nomi
  labs(title = "Distribuzione dei Rating per Genere", x = "Genere", y = "Voto Medio") +
  theme_minimal() +
  theme(legend.position = "none")

df_prof <- df %>% separate_rows(director_professions, sep = ',')
ggplot(df_prof, aes(x = director_professions, y = log(`Worldwide gross $`))) +
  geom_boxplot(outlier.alpha = 0.2) +
  coord_flip() + # Ruota il grafico per leggere meglio i nomi
  labs(title = "Distribuzione dei Rating per Professione del direttore",
       x = "Professione del direttore", y = "Incasso Mondiale") +
  theme_minimal() +
  theme(legend.position = "none")

ggplot(df, aes(x = runtime_minutes, y = `Worldwide gross $`))+
  geom_point(alpha = 0.5)+
  scale_color_gradient2(low = "red", mid = "yellow", high = "green")


# Aggiunta variabili ------------------------------------------------------

df_all <- df
df_all$ROI <- df$`Worldwide gross $`/df$`Production budget $`
df_all$FGR <- df$`Worldwide gross $`*df$`Domestic gross $`/df$`Worldwide gross $`

df_all %>%
  separate_rows(genres, sep = ',') %>%
  mutate(genres = as_factor(genres)) %>%
  count(genres, sort = TRUE) %>%
  print(n = 24)
library(tidyverse)

# 1. Prepariamo i dati filtrati per i generi
df_plot <- df_all %>%
  separate_rows(genres, sep = ',') %>%
  mutate(genres = str_trim(genres)) %>%
  filter(genres %in% c('Action', 'Adventure', 'Animation', 'Comedy', 'Crime'))

# 2. Creiamo il grafico
ggplot(df_plot, aes(x = log(ROI), y = runtime_minutes)) +
  # Disegniamo tutti i punti (divisi per colore dei generi)
  geom_point(aes(colour = genres), alpha = 0.4) +

  # AGGIUNGIAMO IL PUNTO SPECIALE:
  # Filtriamo df_plot al volo per includere solo Mad Max
  geom_point(data = filter(df_plot, movie_title == "Mad Max"),
             color = "black", size = 3, shape = 18) +

  # Aggiungiamo un'etichetta per renderlo leggibile
  geom_text(data = filter(df_plot, movie_title == "Mad Max"),
            aes(label = movie_title),
            vjust = -1, size = 3, fontface = "bold") +

  facet_wrap(~genres) +
  theme_minimal() +
  labs(title = "log(ROI) vs Runtime: Mad Max in evidenza",
       x = "Return on Investment (ROI)",
       y = "Runtime (Minutes)")

# PROBLEMA RAPPRESENTATIVIA' X ESEMPIO MAD MAX COMPARE IN DUE CATEGORIE DI GENERE
# USO IL LOG(ROI) PER MIGLIORARE LA RELAZIONE TRA I MINUTI E IL ROI


which.max(df_all$ROI)
df_all[4335,]
target_directors <- c("Quentin Tarantino")









# ICA ---------------------------------------------------------------------
library(fastICA)
df_num <- df %>%
  select(runtime_minutes, movie_numerOfVotes, `Production budget $`, `Worldwide gross $`) %>%
  scale() # L'ICA richiede dati standardizzati
set.seed(42)
ica_result <- fastICA(df_num, n.comp = 3)

# Aggiungi i risultati al dataframe originale per l'analisi
df$ICA1 <- ica_result$S[,1]
df$ICA2 <- ica_result$S[,2]
df$ICA3 <- ica_result$S[,3]
# Supponendo che le colonne ICA1 e ICA2 siano nel dataframe
ggplot(df, aes(x = ICA1, y = ICA2, color = movie_averageRating)) +
  geom_point(alpha = 0.6) +
  stat_ellipse() + # Aggiunge un'ellisse di confidenza per vedere i raggruppamenti
  scale_color_gradient2(low = "red", mid = "yellow", high = "green", midpoint = 5) +
  labs(title = "Visualizzazione dei Film tramite ICA") +
  theme_bw()
ggplot(df, aes(x = ICA1, y = ICA2, color = movie_averageRating)) +
  geom_point(alpha = 0.6) +
  scale_color_viridis_c() +
  labs(title = "Distribuzione dei Film nello spazio ICA",
       subtitle = "Le componenti cercano sorgenti statisticamente indipendenti",
       x = "Componente Indipendente 1", y = "Componente Indipendente 2")
pca_res <- princomp(df_num)$scores
df$PCA1 <- pca_res[,1]
df$PCA2 <- pca_res[,2]
df$PCA3 <- pca_res[,3]
ggplot(df, aes(x = PCA1, y = PCA2, color = movie_averageRating)) +
  geom_point(alpha = 0.6) +
  scale_color_viridis_c() +
  labs(title = "Distribuzione dei Film nello spazio ICA",
       subtitle = "Le componenti cercano sorgenti statisticamente indipendenti",
       x = "Componente Indipendente 1", y = "Componente Indipendente 2")


# T-SNE -------------------------------------------------------------------


# Regressione -------------------------------------------------------------
library(rsample)
library(tidymodels)

set.seed(1234)
split <- initial_split(df_all)
split
train <- training(split)
test <- testing(split)
test$director_deathYear <- as.numeric(test$director_deathYear)

train_clean <- train %>%
  select(-movie_title, -director_name, -production_date, -director_professions,
         -genres, -FGR)

# 2. Rieseguiamo il modello lineare
lm1 <- lm(`Worldwide gross $` ~ ., data = train_clean)

# 3. Controlliamo il nuovo summary
summary(lm1)


# BACKWARD ----------------------------------------------------------------


step_back <- lm1 %>% stats:::step(direction = "backward")

summary(step_back)

pred.step_back <- predict(step_back, newdata = test)
par(mfrow = c(2,2))
plot(step_back, which = 1:4)

# FORWARD -----------------------------------------------------------------

lm0 <- lm(`Worldwide gross $`~1, data = train_clean)
summary(lm0)
full_model <- formula(lm(`Worldwide gross $` ~ ., train_clean))
step_forw <- lm0 %>% stats::step(direction = "forward", scope = full_model)
summary(step_forw)

pred.step_forw <- predict(step_forw, newdata = test)

par(mfrow = c(2,2))
plot(step_forw, which = 1:4)


# Best subset selection ---------------------------------------------------

library(leaps)
library(tidyverse)

# Esecuzione della Best Subset Selection
# nvmax dovrebbe essere almeno pari al numero di colonne del tuo matrix x
bss_model <- regsubsets(`Worldwide gross $` ~ .,
                        data = train_clean,
                        nvmax = 20) # Aumenta se hai più di 20 variabili

# Estrazione dei risultati
bss_summary <- summary(bss_model)
bss_summary
# Creiamo un tibble con i criteri di valutazione
results <- tibble(
  n_vars = 1:length(bss_summary$rss),
  adjr2  = bss_summary$adjr2,
  cp     = bss_summary$cp,
  bic    = bss_summary$bic
)

# Plot dell'Adjusted R2 (Cerchiamo il massimo)
ggplot(results, aes(x = n_vars, y = adjr2)) +
  geom_line() +
  geom_point() +
  geom_point(data = results %>% filter(adjr2 == max(adjr2)), color = "red", size = 3) +
  labs(title = "Miglior Adjusted R-squared", x = "Numero di Variabili", y = "Adj R2")

# Plot del BIC (Cerchiamo il minimo)
ggplot(results, aes(x = n_vars, y = bic)) +
  geom_line() +
  geom_point() +
  geom_point(data = results %>% filter(bic == min(bic)), color = "red", size = 3) +
  labs(title = "Miglior BIC", x = "Numero di Variabili", y = "BIC")

# Visualizzazione a griglia (molto utile)
plot(bss_model, scale = "bic")


ERROR: Error in library(patchwork): there is no package called ‘patchwork’
